# ⚡ MLM 10M Staged-Node Growth Trainer (Google Colab T4 GPU)

This notebook trains our **10.8M-parameter Deep Hybrid Liquid-Delta Graph MLM (`v2_hybrid`)** using our **3-Stage Recombinative Node Growth** architecture:
1. **Stage 1 (`english_learn` — 4 Specialist Graph Nodes):** Streams Simple English Wikipedia, SciQ, and SQuAD passages via Rust `polars` and pretrains on continuous English paragraphs (`mask = 1.0` across all tokens) so the Core Thinking + English Grammar nodes master fluent English first.
2. **Stage 2 (`english_understand` — Grow `4 → 6` Specialist Graph Nodes via Genetic Crossover):** Spawns +2 Reading Comprehension nodes from the strongest Stage 1 parent nodes (`mature_experts = 4` protected at `0.15x` LR) and trains on SQuAD + SciQ + Wikipedia QA interleaved with 25% Stage 1 English replay.
3. **Stage 3 (`english_chat` — Grow `6 → 8` Specialist Graph Nodes = `10.8M` Params via Genetic Crossover):** Spawns +2 Dialogue/Chat nodes (`mature_experts = 6` protected at `0.15x` LR) and trains on SmolTalk Everyday Conversations + Dolly-15k + Alpaca interleaved with 25% Stage 1 & 2 replay.
4. **Exports Both Formats:**
   - `runs/mlm-10m-colab/mlm_10m_v2.mlm2` (SafeTensors + Rust BPE Tokenizer for `mlm chat` / `mlm-chat`)
   - `runs/mlm-10m-colab/mlm_10m_v2.gguf` (`llama.cpp`-compatible GGUF v3 file)

In [ ]:
# 1. Clone / Update MLM Repository & Install Rust-Backed Dependencies on Colab GPU
import os
import subprocess

REPO_URL = "https://github.com/aswin402/mlm.git"
if not os.path.exists("/content/mlm"):
    subprocess.run(["git", "clone", REPO_URL, "/content/mlm"], check=True)
%cd /content/mlm
!git pull origin main
!pip install -q -U "jax[cuda12]" flax optax pydantic numpy tokenizers polars safetensors
!pip install -q -e . --no-deps

import jax
print("JAX devices:", jax.devices())

In [ ]:
# 2. Run 3-Stage Node Growth (4 Learn -> 6 Understand -> 8 Chat = 10.8M Params, 5000 GPU Steps)
!python scripts/train_10m_english_colab.py \
    --data-root data/curated-english-v1 \
    --out runs/mlm-10m-colab \
    --max-rows-per-source 4000 \
    --s1-steps 1500 \
    --s2-steps 1500 \
    --s3-steps 2000 \
    --eval-every 250 \
    --batch-size 32 \
    --seq-len 128

In [ ]:
# 3. Interactive Test & Download Trained 10M Checkpoint (.mlm2) and llama.cpp (.gguf)
!python -m mlm.cli chat runs/mlm-10m-colab/mlm_10m_v2.mlm2 --message "Hi! How are you?"
!python -m mlm.cli chat runs/mlm-10m-colab/mlm_10m_v2.mlm2 --message "What is science?"

try:
    from google.colab import files
    files.download("runs/mlm-10m-colab/mlm_10m_v2.mlm2")
    files.download("runs/mlm-10m-colab/mlm_10m_v2.gguf")
except Exception as e:
    print("Colab download skipped (not running in browser):", e)